In [ ]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)

In [ ]:
# Paths -- edit these for your environment.
RAW_DIR  = "/PATH/TO/raw"              # raw MGB extracts (edw/, clarity/, Datamart/)
DATA_DIR = "/PATH/TO/data"             # project data root; lookup tables live here
WORK_DIR = f"{DATA_DIR}/20250604"      # identified intermediate outputs -- keep on the secure server
DEID_DIR = f"{WORK_DIR}/deidentified"  # de-identified outputs and ID / date-shift maps

In [ ]:
microbiology_cultures_cohort = pd.read_csv(f'{WORK_DIR}/microbiology_cultures_cohort.csv')
# med_data = pd.read_csv(f'{RAW_DIR}/edw/medications/MGB_abx_courses_20150220-20240715.csv')
med_data = pd.read_csv(f'{RAW_DIR}/edw/medications/MGB_abx_20150323-20240715.csv')

In [ ]:
base_cohort = microbiology_cultures_cohort[['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc']].drop_duplicates()
base_cohort

In [ ]:
med_data

In [ ]:
# Step 3: Join with base cohort and calculate time difference
med_data = med_data.rename(columns = {'patientID':'anon_id',
                                      # 'start_DT':'medicationOrderDTS',
                                      'order_DTS':'medicationOrderDTS',
                                      'antibiotic':'medication_name'})
merged = base_cohort.merge(med_data,
                           on = 'anon_id',
                           how = 'left')
merged['medication_time_to_cultureTime'] = (pd.to_datetime(merged['order_time_jittered_utc']) - pd.to_datetime(merged['medicationOrderDTS'])).dt.days
# merged['medication_time_to_cultureTime'] = (pd.to_datetime(merged['order_time_jittered_utc']) - pd.to_datetime(merged['medicationOrderDTS']).dt.tz_localize('UTC')).dt.days
merged

In [ ]:
# TODO: Clean-up Mapping File
# Step 4: Map valid antibiotics
antibiotic_mapping = {
    'CEFTRIAXONE': 'CEF2',  # Assuming this is equivalent to Ceftriaxone
    'CIPROFLOXACIN': 'CIP3',  # Mapping to Ciprofloxacin
    'AMPICILLIN-SULBACTAM': None,  # No corresponding code in the old map
    'VANCOMYCIN': 'VAN',
    'METRONIDAZOLE': 'MET1',  # Assuming standard Metronidazole
    'CEFAZOLIN': 'CEF',
    'NITROFURANTOIN': 'NIT',
    'AMPICILLIN': 'AMP1',
    'LINEZOLID': 'LIN',
    'CEFEPIME': 'CEF6',
    'TRIMETHOPRIM-SULFAMETHOXAZOLE': 'SUL',  # Same as Sulfamethoxazole-Trimethoprim
    'LEVOFLOXACIN': 'LEV',
    'PIPERACILLIN-TAZOBACTAM': 'PIP1',
    'MINOCYCLINE': 'MIN',
    'DOXYCYCLINE': 'DOX',
    'CEFPODOXIME': 'CEF3',
    'FOSFOMYCIN': 'FOS',
    # Skip None (if an entry is None, it remains so)
    None: None,
    'MEROPENEM': 'MER',
    'AZITHROMYCIN': 'AZI',
    'PENICILLIN': 'PEN',
    'CEFUROXIME': 'CEF7',
    'AMOXICILLIN-CLAVULANATE': 'AMO',
    'CEFTAZIDIME': 'CEF10',
    'GENTAMICIN': 'GEN1',
    'DAPTOMYCIN': None,  # Not in the old mapping (likely Daptomycin)
    'CEPHALEXIN': 'CEP',
    'CLINDAMYCIN': 'CLI',
    'NAFCILLIN': None,  # Not in the old mapping
    'AZTREONAM': 'AZT1',
    'IMIPENEM': None,  # Not in the old mapping
    'ERTAPENEM': 'ERT',
    'CEFADROXIL': 'CEF8',
    'AMOXICILLIN': 'AMO1',
    'CEFOXITIN': 'CEF5',
    'OXACILLIN': None,  # Not in the old mapping
    'CLARITHROMYCIN': 'CLA',
    'AMIKACIN': 'AMI',
    'MOXIFLOXACIN': 'MOX',
    'TETRACYCLINE': None,  # Not in the old mapping
    'TEDIZOLID': 'TED',
    'CEFTOLOZANE-TAZOBACTAM': None,  # Not in the old mapping
    'TRIMETHOPRIM': 'TRI',
    'BACITRACIN': None,  # Not in the old mapping
    'CEFTAZIDIME-AVIBACTAM': None,  # Not in the old mapping
    'CEFDINIR': 'CEF4',
    'DICLOXACILLIN': 'DIC',
    'CEFTAROLINE': None,  # Not in the old mapping
    'DALBAVANCIN': None,  # Not in the old mapping
    'FIDAXOMICIN': 'FID',
    'TOBRAMYCIN': 'TOB',
    'TIGECYCLINE': None,  # Not in the old mapping
    'DEMECLOCYCLINE': None,  # Not in the old mapping
    'CEFIDEROCOL': None,  # Not in the old mapping
    'ORITAVANCIN': None,  # Not in the old mapping
    'OMADACYCLINE': None,  # Not in the old mapping
    'CEFIXIME': None,  # Not in the old mapping
    'CEFOTETAN': None,  # Not in the old mapping
    'TELAVANCIN': None,  # Not in the old mapping
    'MEROPENEM-VABORBACTAM': None,  # Not in the old mapping
    'POLYMYXIN B': None,  # Not in the old mapping (Colistin exists but Polyymyxin B is different)
    'IMIPENEM-RELEBACTAM': None,  # Not in the old mapping
    'CEFACLOR': None,  # Not in the old mapping
    'ERAVACYCLINE': None,  # Not in the old mapping
    'CEFOTAXIME': None,  # Not in the old mapping
    'CHLORAMPHENICOL': None,  # Not in the old mapping
    'VANCOMYCINC': 'VAN',  # Assuming a typo for Vancomycin
    'STREPTOMYCIN': None,  # Not in the old mapping
    'DELAFLOXACIN': None,  # Not in the old mapping
    'NEOMYCIN-POLYMYXIN': None,  # Not in the old mapping
    'NOVOBIOCIN': None,  # Not in the old mapping
    'LEFAMULIN': None,  # Not in the old mapping
    'CEFPROZIL': None  # Not in the old mapping
}


In [ ]:
merged['medication_category'] = merged['medication_name'].map(antibiotic_mapping)
# valid_antibiotics = merged.dropna(subset=['medication_category'])

In [ ]:
final_table = merged[merged['medication_time_to_cultureTime'] > 0]

group_cols = ["anon_id", "pat_enc_csn_id_coded", "order_proc_id_coded", 
              "order_time_jittered_utc","medicationOrderDTS", 
              "medication_time_to_cultureTime", "medication_name", 
              "medication_category"]

final_table = final_table.groupby(group_cols, as_index=False).first()

final_table = final_table.sort_values(by=group_cols)

final_table

In [ ]:
final_table_clean = final_table[['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 
                                 'order_time_jittered_utc','medication_name','medicationOrderDTS',
                                 'medication_time_to_cultureTime','medication_category']]

final_table_clean


In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns

final_table_clean['medication_name'] = final_table_clean['medication_name'].fillna('Missing')
top10_var1 = final_table_clean['medication_name'].value_counts().nlargest(10).index
data_filtered = final_table_clean[final_table_clean['medication_name'].isin(top10_var1)]

plt.plot()
sns.countplot(x='medication_name',
              data = data_filtered,
              order=top10_var1)
plt.xticks(rotation=90)

In [ ]:
final_table_clean.to_csv(f'{WORK_DIR}/microbiology_cultures_prior_med_augmented.csv',
                         index = False)

## Deidentification

In [ ]:
# Load the saved mappings (assume you saved as CSVs before)
anon_map_df = pd.read_csv(f'{DEID_DIR}/anon_id_map.csv')
enc_map_df = pd.read_csv(f'{DEID_DIR}/enc_id_map.csv')
order_proc_map_df = pd.read_csv(f'{DEID_DIR}/order_proc_id_map.csv')

In [ ]:
# Convert to Dict
anon_id_map = dict(zip(anon_map_df['anon_id'], anon_map_df['anon_id_deid']))
enc_id_map = dict(zip(enc_map_df['pat_enc_csn_id_coded'], enc_map_df['pat_enc_csn_id_deid']))
order_proc_map = dict(zip(order_proc_map_df['order_proc_id_coded'], order_proc_map_df['order_proc_id_deid']))

# Apply the mappings
final_table_deid = final_table_clean.copy()
final_table_deid['anon_id'] = final_table_deid['anon_id'].map(anon_id_map)
final_table_deid['pat_enc_csn_id_coded'] = final_table_deid['pat_enc_csn_id_coded'].map(enc_id_map)
final_table_deid['order_proc_id_coded'] = final_table_deid['order_proc_id_coded'].map(order_proc_map)

final_table_deid

In [ ]:
final_table_deid.to_csv(f'{DEID_DIR}/microbiology_cultures_prior_med_augmented.csv',
                         index = False)